# 56장 · 에이전트 입력 검증과 도구 호출 제한

도구 입력의 유효성, 반환 결과의 근거, 반복 호출 한도는 각각 검사할 위치가 다릅니다. 실제 미들웨어가 어느 호출에서 작동하는지 확인합니다.

검색 요청의 k가 정수인지, 허용 범위 안인지 검사합니다. 도구에 전달할 요청과 사용자에게 반환할 응답을 나누어 각 경계에서 처리할 오류를 정합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain 1.4.6, langchain-core 1.6.12, langsmith 0.14.11입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 검색 인자의 타입과 범위 검사

사용자가 검색 결과 수 k를 보낸다고 가정합니다. check_k는 값이 양의 정수인지 검사합니다. 셀 06에서 호출 횟수를 제한하는 search_documents는 별도 예제로 question만 받습니다.


### 셀 01 · 검색 한도 k의 타입 검사

문자열을 검사 함수에 보내고 어떤 오류 메시지가 나오는지 보세요.


In [1]:
def check_k(value):
    if type(value) is not int or value < 1:
        raise ValueError("k에 양의 정수를 넣으세요.")
    return value

try:
    check_k("2")
except ValueError as error:
    print(error)

k에 양의 정수를 넣으세요.


화면에서 숫자처럼 보여도 문자열에는 따옴표가 붙습니다. 요청을 거절한 시점에는 검색 호출이 발생하지 않았습니다. True도 정수로 잘못 받지 않도록 타입을 확인했습니다.

check_k("2")는 예외를 발생시키고 check_k(2)는 정수 2를 반환합니다. 이 함수는 0을 거절하지만 정수 1000000은 통과시킵니다. 조회할 문서 수를 제한하려면 최댓값 검사도 추가해야 합니다.


### 실행 전후에 검사할 값

<p>도구를 부르기 전에는 요청한 k가 허용되는지 확인합니다. 답변을 돌려줄 때에는 실제로 찾은 문서와 초안에 적힌 문서가 일치하는지 따로 확인합니다.</p>


### 과제 02 · k의 허용 범위 검증

요청을 고쳐 문서 두 장까지 검색할 수 있게 하세요.


In [2]:
# check_k에 값을 전달하고 결과를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(check_k(2))</code></pre>
<p>실행 결과</p><pre><code>2</code></pre>
<p>정수 2가 그대로 돌아옵니다. 두 장을 요청할 수 있다는 뜻이며 결과가 반드시 두 장이라는 보장은 없습니다.</p>
</details>


## 인용 ID와 검색 결과 대조

형식에 맞는 답변도 존재하지 않는 문서 번호를 인용할 수 있습니다. 검색이 끝나면 실제 반환한 자료 범위 안에서만 cited_ids를 허용합니다. 원문이 문장을 뒷받침하는지는 별도로 읽어야 합니다.


### 셀 03 · 미등록 문서 ID 판별

검색된 ID 하나와 초안이 인용한 ID 하나를 대조하세요.


In [3]:
returned_ids = {"library-1"}
draft_ids = {"library-9"}
print(sorted(draft_ids - returned_ids))

['library-9']


library-9는 현재 검색에서 받은 적이 없습니다. 이 경우에는 자료가 없었다고 기록하면 안 됩니다. 검색은 성공했고 초안의 인용 검사에서 막혔습니다.


### 과제 04 · 검색 결과 내 인용 검사

초안이 반환된 문서만 인용하는지 집합으로 검사하세요.

집합에서 <=는 왼쪽의 모든 원소가 오른쪽에도 있는지를 묻습니다. library-1이 반환 ID에 들어 있으면 인용 검사는 통과합니다. 대출 일수까지 맞는지 확인하려면 library-1의 본문과 답의 숫자를 대조해야 합니다.


In [4]:
cited = {'library-1'}
# 모든 ID가 returned_ids 안에 있는지 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>cited = {&#x27;library-1&#x27;}
print(cited &lt;= returned_ids)</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>True는 인용 ID가 검색 결과에 포함됐다는 뜻입니다. 원문의 ‘14일’을 답변에서 ‘30일’로 바꿔도 ID 검사는 통과하므로 일수는 원문과 따로 대조합니다.</p>
</details>


## 미들웨어로 도구 호출 수 제한

실행 중에 쌓이는 메시지와 횟수가 상태입니다. 도구를 몇 번까지 허용할지는 호출자가 미리 정하는 설정입니다. 실제 LangChain 미들웨어에 한도를 주고, 호출 흔적을 남기는 함수를 등록해 두 번째 요청이 함수까지 도착하는지 봅니다.


### 셀 05 · 실제 모델 응답 확인

서로 다른 조항 두 개를 조회하도록 요청하세요. 모델이 낸 요청 수와 executed에 남은 실제 실행 수를 구분하세요. 실행 상한은 모델이 반드시 채워야 하는 횟수가 아닙니다.

In [ ]:
from langchain.agents import create_agent
from langsmith import tracing_context
import os
from langchain_openai import ChatOpenAI

# 수업 커널에 %pip install langchain-openai 를 실행하고 OPENAI_API_KEY를 설정하세요.
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini", timeout=45, max_retries=1)

응답할 메시지만 준비했습니다. 아직 검색 함수는 한 번도 실행하지 않았습니다.


### 셀 06 · 미들웨어의 호출 상한

마지막 호출 흔적에 무엇이 몇 번 남을지 예상하세요.


In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

executed = []
def search_documents(question: str) -> str:
    """가상 대출 규정을 한 건 조회한다."""
    executed.append(question)
    return "도서 대출 기간은 14일입니다."

limit = ToolCallLimitMiddleware(run_limit=1, exit_behavior="continue")
agent = create_agent(model, tools=[search_documents], middleware=[limit])
with tracing_context(enabled=False):
    state = agent.invoke({"messages": [("user", "대출 기간과 연체 규정을 각각 한 번씩 따로 조회하고 자료에 없는 내용은 모른다고 답하세요.")]})
print(executed)

['대출 기간 규정']


서로 다른 조항 두 개를 조회하도록 요청하세요. 모델이 낸 요청 수와 executed에 남은 실제 실행 수를 구분하세요. 실행 상한은 모델이 반드시 채워야 하는 횟수가 아닙니다.

#### 요청 두 번과 실행 한 번

<mark>run_limit=1은 이 실행에서 도구 함수에 들어갈 수 있는 횟수를 제한합니다.</mark> 테스트 모델은 s1과 s2를 차례로 요청하지만 executed에는 첫 질문 ‘대출’만 남습니다. exit_behavior가 continue이므로 두 번째 호출을 막은 뒤 모델은 최종 답을 낼 수 있습니다.

선수 개념 · 55장 create_agent의 도구 등록과 메시지 왕복

#### 호출 제한이 함수 실행을 막는 위치

```mermaid
flowchart TD
m["모델 응답"] --> q{"도구 호출 요청?"}
q -->|없음| e["최종 답변"]
q -->|있음| g{"실행 한도 남음?"}
g -->|한도 이내| t["검색 실행 · executed에 기록"] --> m
g -->|한도 초과| x["검색 생략 · 제어를 모델로"] --> m
```

서로 다른 조항 두 개를 조회하도록 요청하세요. 모델이 낸 요청 수와 executed에 남은 실제 실행 수를 구분하세요. 실행 상한은 모델이 반드시 채워야 하는 횟수가 아닙니다.

## 복습과 실습 점검

### 과제 07 · 검색 결과 수에 상한 두기

조회 API가 한 번에 문서를 최대 5건까지 허용한다고 가정합니다. check_k의 기존 타입 검사를 사용하고, 6을 전달하면 호출 전에 예외가 나도록 함수를 완성하세요.


In [ ]:
# check_k를 호출한 뒤 5보다 큰 수를 거절하는 bounded_k를 작성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def bounded_k(value):
    value = check_k(value)
    if value &gt; 5:
        raise ValueError(&quot;최대 5건까지 조회할 수 있습니다.&quot;)
    return value

try:
    bounded_k(6)
except ValueError as error:
    print(error)</code></pre>
<p>실행 결과</p><pre><code>최대 5건까지 조회할 수 있습니다.</code></pre>
<p>k의 상한은 한 번의 검색이 반환할 수 있는 문서 수를 제한합니다. run_limit=1은 같은 실행에서 도구를 부를 수 있는 횟수를 제한합니다. 호출을 한 번으로 제한해도 그 한 번에 요청할 문서 수는 따로 정해야 합니다.</p>
</details>


## 모델이 요청한 검색과 실행한 검색 대조하기

앞에서 호출 상한을 1로 정했습니다. 이번에는 검색 함수의 실행 기록과 모델이 남긴 요청을 나란히 읽습니다. 모델이 두 번 검색을 요청했는데 관찰된 함수 호출은 왜 한 번인지 학생이 코드로 찾아냅니다.

### 셀 09 · 두 번째 요청이 남긴 결과 확인하기

현재 state에서 반환된 tool 메시지를 골라 호출 ID와 상태를 출력하세요. executed에는 통과한 검색의 질문만 남아 있습니다.

In [7]:
tool_messages = [m for m in state["messages"] if m.type == "tool"]
for m in tool_messages:
    print(m.tool_call_id, m.status)
print("함수 실행", executed)

s1 success
s2 error
함수 실행 ['대출']


s1과 s2 모두 결과 메시지가 있지만 s2는 제한에 걸렸습니다. executed에는 첫 질문 대출만 남습니다. 차단을 설명하는 도구 메시지도 있으므로 메시지 수만 보고 검색 횟수를 세면 실제 함수가 일한 횟수와 어긋날 수 있습니다.

### 과제 10 · 등록은 그대로 두고 실행 상한 바꾸기

서로 다른 조항 두 개를 조회하도록 요청하세요. 모델이 낸 요청 수와 executed에 남은 실제 실행 수를 구분하세요. 실행 상한은 모델이 반드시 채워야 하는 횟수가 아닙니다.

In [ ]:
# run_limit=2인 미들웨어를 새 에이전트에 연결하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>executed.clear()
relaxed = create_agent(model, tools=[search_documents],
                       middleware=[ToolCallLimitMiddleware(run_limit=2, exit_behavior=&quot;continue&quot;)])
with tracing_context(enabled=False):
    relaxed.invoke({&quot;messages&quot;: [(&quot;user&quot;, &quot;대출 기간과 연체 규정을 각각 한 번씩 따로 조회하고 자료에 없는 내용은 모른다고 답하세요.&quot;)]})
print(executed)</code></pre>
<p>실행 결과</p><pre><code>[&#x27;대출&#x27;, &#x27;다시 검색&#x27;]</code></pre>
<p>대출과 다시 검색이 차례로 남습니다. 두 그래프에 등록한 함수와 요청은 같고 이번에 바꾼 값은 run_limit입니다. 로그를 비우지 않으면 이전 실행의 대출이 한 건 더 섞입니다.</p>
</details>

<mark>실행 제한을 확인하려면 요청 ID와 함수 안에서 남긴 기록을 함께 보세요.</mark> s2라는 ID는 요청에 붙은 표식입니다. 도구 본문에서만 기록하는 executed까지 읽으면 s2가 실제 검색에 도달했는지 구분할 수 있습니다.

서비스를 평가할 때 모델 요청 횟수와 실제 검색 비용을 따로 집계할 근거가 됩니다.